# Logistic regression từ đầu: gradient descent và Newton trên review Olist

Câu hỏi kinh doanh: đơn giao trễ bao lâu thì khách cho review 1–2 sao? Câu hỏi kỹ thuật: tự viết
logistic regression, chứng minh bằng số rằng loss lồi (Hessian nửa xác định dương), so tốc độ
gradient descent và Newton, và hiểu khi nào regularization là bắt buộc chứ không chỉ để chống
overfit.

Định nghĩa dùng chung của track: đơn giao xong là `order_status == "delivered"` và có
`order_delivered_customer_date`; mỗi đơn giữ review có `review_answer_timestamp` mới nhất; review
xấu là `review_score <= 2`; giao trễ là so nguyên timestamp
`order_delivered_customer_date > order_estimated_delivery_date`. Chia theo thời điểm mua: train trước
2018-01-01, val tới trước 2018-04-01, test tới hết 2018-08-31; bỏ đơn mua từ 2018-09-01.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` là chỗ phân tích tự do trên dữ liệu thật; test bỏ qua chúng.
- Viết báo cáo vào `REPORT.md` cạnh notebook.
- Chấm: `uv run pytest tracks/data-science/modules/01-math-for-ml/exercises/advanced/02-logistic-newton -q`

In [ ]:
import numpy as np
import pandas as pd

## 1. Bảng dữ liệu

`build_review_table(orders, reviews)`: mỗi dòng một đơn giao xong có review, cột `order_id`, `bad`
(0/1), `late` (0/1), `delay_days` (ngày giao thực tế trừ ngày hẹn, số thực, âm là giao sớm),
`delivery_days` (ngày giao trừ ngày mua, số thực), `split`.

In [ ]:
def build_review_table(orders: pd.DataFrame, reviews: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## 2. Loss, gradient, Hessian

`X` đã có cột intercept (cột 0). Với $$z = Xw$$, $$p = \sigma(z)$$:

$$
L(w) = \frac{1}{n}\sum_i \left[\log(1 + e^{z_i}) - y_i z_i\right] + \frac{\lambda}{2}\sum_{j \ge 1} w_j^2
$$

Trả `(loss, grad, hess)`. Không phạt intercept. Phải ổn định số khi $$|z|$$ lớn.

In [ ]:
def logistic_loss_grad_hess(w: np.ndarray, X: np.ndarray, y: np.ndarray, l2: float = 0.0):
    # TODO: loss, gradient (1/n) Xᵀ(p − y) + phần l2, Hessian (1/n) Xᵀ diag(p(1 − p)) X + phần l2
    raise NotImplementedError

## 3. Hai optimizer

- `fit_logistic_gd(X, y, lr, n_iter, l2=0.0) -> (w, losses)`: w khởi tạo 0, lr cố định,
  `losses` dài `n_iter + 1` (loss tại w0 và sau mỗi bước).
- `fit_logistic_newton(X, y, n_iter=25, tol=1e-10, l2=0.0) -> (w, losses)`: w khởi tạo 0,
  mỗi bước giải $$H \Delta = \nabla L$$ (không nghịch đảo H) rồi $$w \leftarrow w - \Delta$$; dừng khi
  $$\lVert\nabla L\rVert < \text{tol}$$. `losses` gồm loss tại w0 và sau mỗi bước đã làm.

In [ ]:
def fit_logistic_gd(X: np.ndarray, y: np.ndarray, lr: float, n_iter: int, l2: float = 0.0):
    # TODO
    raise NotImplementedError


def fit_logistic_newton(X: np.ndarray, y: np.ndarray, n_iter: int = 25, tol: float = 1e-10, l2: float = 0.0):
    # TODO
    raise NotImplementedError

## 4. Phân tích tự do

Gợi ý các câu hỏi cho `REPORT.md`: tốc độ hội tụ hai optimizer trên dữ liệu thật (số bước, thời
gian mỗi bước, khi nào Newton quá đắt); ý nghĩa hệ số; chất lượng trên test; vì sao bài toán này lồi
còn một mạng nơ-ron nhỏ trên cùng dữ liệu thì không (thử nhiều seed); dữ liệu tách được thì sao.

In [ ]:
# explore: đọc Olist thật
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
raw = ROOT / "data" / "raw"
dates = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=dates)
reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
table = build_review_table(orders, reviews)
table.groupby("split")[["bad", "late"]].agg(["mean", "size"])